# Proyecto Final Optimización Optimización costos en rutas detransporte MEGABUS

#### Mateo Gutiérrez Melo - mgutierrezca@unal.edu.co
#### Fabián Leandro López - flopezgo@unal.edu.co
#### Carlos Jiménez Camargo - cajimenezca@unal.edu.co


### Instalación de librerias 

A continuación se presenta la instalación de las librerias necesarias para visualizar, entender y optimizar los costos de las rutas de MEGABUS

In [200]:
## Instalación de paquetes

%pip install -q -U pandas
%pip install -q -U matplotlib

%pip install -q -U folium
%pip install -q -U geopandas
%pip install -q -U  osmnx 
%pip install scikit-learn
%pip install -q -U  networkx
%pip install shapely
%pip install matplotlib


[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To 

#### Importación de librerias

In [201]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
import osmnx as ox
import networkx as nx
from shapely import wkt
import os
import gurobipy as gp
from gurobipy import GRB
from geopy.distance import geodesic


## Lectura de los datos para las estaciones de MEGABUS


### Visualización de las rutas en Pereira 

Para efectos practicos primero se va a graficar el mapa de la ciudad de Pereira vacio y posteriormente se integrarán las 24 rutas correspondientes al servicio de transporte que ofrece la empresa MEGABUS

In [202]:
# Creación Mapa de Pereira
mapa_pereira = folium.Map(location=[4.81, -75.70], zoom_start=13)

mapa_pereira


#### Unión de rutas MEGABUS

Para integrar las rutas de MEGABUS en el mapa de Pereira es necesario leer los datos y darles un formato de referencia geoespacial tal y como lo muestra el siguiente script

In [203]:
# Obtención de todos los archivos que contengan las rutas 
archivos = os.listdir('./data/')
rutas = [archivo for archivo in archivos if archivo.startswith('Ruta')]

map_df_rutas = []

# Iteración sobre cada archivo de rutas
for ruta in rutas : 

    df_temp = pd.read_csv(f'./data/{ruta}')
    df_temp['the_geom'] = df_temp['the_geom'].apply(wkt.loads)
    
    # Transformación del DataFrame a un GeoDataFrame
    gdf_temp = gpd.GeoDataFrame(df_temp, geometry='the_geom', crs="EPSG:4326")

    ## Agregar el GeoDataFrame al mapa de GeoDataFrames de rutas
    map_df_rutas.append(gdf_temp)

In [204]:
colors = ['#1f77b4','#1f56c4', '#aec7e8', '#ff7f0e', '#ffbb78', '#2ca02c', '#98df8a', '#d62728', '#ff9896', '#9467bd', '#c5b0d5', '#8c564b', '#c49c94', '#e377c2', '#f7b6d2', '#7f7f7f', '#c7c7c7', '#bcbd22', '#dbdb8d', '#17becf', '#9edae5', '#393b79', '#5254a3', '#6b6ecf', '#9c9ede']


A continuación se muestran las rutas de manera gráfica en un formato de imagen para poder ver los lugares por donde pasa cada bus

In [205]:
# Crear una leyenda de colores
legend_html = '''
<div style="
    position: fixed; 
    bottom: 50px; 
    left: 50px; 
    width: 120px; 
    height: 150px; 
    border:2px solid grey; 
    z-index:9999; 
    font-size:14px;
    overflow-y: scroll;
">
    <p>Colores de Rutas:</p>
'''

for i, color in enumerate(colors):
    legend_html += f'<p><span style="color:{color};">■</span> Ruta {i+1}</p>'

legend_html += '</div>'

mapa_pereira.get_root().html.add_child(folium.Element(legend_html))

cont = 0
for gdf in map_df_rutas:
    for _, row in gdf.iterrows():
        # Extraer las coordenadas de la geometría
        coords = list(row['the_geom'].coords)

        # Crear una línea para cada geometría con un color de la lista
        folium.PolyLine(
            locations=[[coord[1], coord[0]] for coord in coords], 
            color=colors[cont % len(colors)],
        ).add_to(mapa_pereira)
    cont += 1

mapa_pereira

## Cálculo de distancias para cada ruta

A continuación se muestra como se calculas las distancias de cada ruta usando los datos georeferenciales

In [206]:
from geopy.distance import great_circle

# Proyectar un CRS (Coordinate Reference System) para el mapa de rutas en Pereira 
gdf = gdf.to_crs("EPSG:3116")

map_distancias_rutas = []

for ruta in map_df_rutas: 
    coords = list(ruta.geometry.iloc[0].coords)

    total_distance = 0

    # Cálculo de distancia sumando las distancias entre cada par de puntos
    for i in range(len(coords)-1):
        point1 = (coords[i][1], coords[i][0])  # (lat, lon)
        point2 = (coords[i + 1][1], coords[i + 1][0])
        distance = great_circle(point1, point2).kilometers
        total_distance += distance
    
    map_distancias_rutas.append(total_distance)

In [207]:
map_distancias_rutas

[9.214113799581524,
 20.36396278472651,
 7.124755062446784,
 4.287462484577862,
 7.091237956704591,
 57.412572540744605,
 6.299054155925464,
 5.800077198294771,
 8.63721511393922,
 4.67473640117491,
 6.254473510251086,
 5.462582742946204,
 6.635824235239598,
 11.626910885310469,
 5.425676374603256,
 9.870964421606251,
 21.28044972947213,
 7.655192274079613,
 3.4608150950657546,
 45.622143342197255,
 5.790989209460155,
 32.57598386039721,
 17.314709341669342,
 8.894315893023316,
 7.0139912141771275]

### Lectura de archivos MEGABUS

A continuación se presenta la lectura de los archivos para conocer la demanda promedio por intervalo de tiempo para cada ruta

In [208]:
df_datos_megabus = pd.read_csv("./data/DatosDemandaBusesIntervaloHora.csv")

df_mañana = df_datos_megabus[df_datos_megabus['Intervalo de Hora'] == 'Mañana']
df_tarde = df_datos_megabus[df_datos_megabus['Intervalo de Hora'] == 'Tarde']
df_noche = df_datos_megabus[df_datos_megabus['Intervalo de Hora'] == 'Noche']


In [209]:
demanda_intervalos = {
    0: df_mañana.set_index('Rutas')['Demanda'].to_dict(),
    1: df_tarde.set_index('Rutas')['Demanda'].to_dict(),
    2: df_noche.set_index('Rutas')['Demanda'].to_dict()
}



In [210]:
intervalo_demandas = {
    0: df_mañana.set_index('Rutas')['Demanda'].to_dict(),
    1:  df_tarde.set_index('Rutas')['Demanda'].to_dict(),
    2:  df_noche.set_index('Rutas')['Demanda'].to_dict()
}


In [211]:
import gurobipy as gp
from gurobipy import GRB

# Datos de entrada
coords_rutas = map_df_rutas
distancias = map_distancias_rutas

# Otros parámetros
capacidad_A = 30 # Capacidad Total bus tipo A
capacidad_B = 50 # Capacidad Total bus tipo B
costo_fijo_A = 800000 # Costos Mensuales 
costo_fijo_B = 1300000 # Costos Mensuales
consumo_A = 1.5  # litros/km
consumo_B = 3  # litros/km
costo_combustible = 4102 # $/litro
max_buses_A = 187
max_buses_B = 110

# Crear el modelo
model = gp.Model("Optimización Dinámica de Rutas de MEGABUS")

# Variables de decisión
num_rutas = 25
intervalo_horas = 3
costo_intervalo_A = costo_fijo_A / 90
costo_intervalo_B = costo_fijo_B / 90

x = model.addVars(num_rutas, intervalo_horas, vtype=GRB.INTEGER, name="x")  # Número de buses Tipo A
y = model.addVars(num_rutas, intervalo_horas, vtype=GRB.INTEGER, name="y")  # Número de buses Tipo B

# Función objetivo
model.setObjective(
    gp.quicksum(
        x[i, t] * (costo_intervalo_A + distancias[i] * consumo_A * costo_combustible) +
        y[i, t] * (costo_intervalo_B + distancias[i] * consumo_B * costo_combustible)
        for i in range(num_rutas) for t in range(intervalo_horas)
    ), 
    GRB.MINIMIZE
)

# Restricciones de demanda por intervalo
for intervalo in range(intervalo_horas):
    for i in range(num_rutas):
        if i in demanda_intervalos[intervalo]:
            model.addConstr(
                x[i, intervalo] * capacidad_A + y[i, intervalo] * capacidad_B >= demanda_intervalos[intervalo][i]
            )

# Restricciones de disponibilidad de buses por día
model.addConstr(gp.quicksum(
    x[i, t] for i in range(num_rutas) for t in range(intervalo_horas)
) <= max_buses_A)

model.addConstr(gp.quicksum(
    y[i, t] for i in range(num_rutas) for t in range(intervalo_horas)
) <= max_buses_B)

# Optimizar el modelo
model.optimize()

# Verifica que el modelo se haya optimizado
if model.status == GRB.OPTIMAL:
    # Mostrar resultados
    for i in range(num_rutas):
        for intervalo in range(intervalo_horas):
            valor_x = x[i, intervalo].x
            valor_y = y[i, intervalo].x
            print(f"Ruta {i+1}, Intervalo {intervalo}: Buses Tipo A = {valor_x}, Buses Tipo B = {valor_y}")
else:
    print("El modelo no se ha optimizado correctamente.")

Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (mac64[arm] - Darwin 23.4.0 23E224)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Optimize a model with 74 rows, 150 columns and 294 nonzeros
Model fingerprint: 0xc68e47cf
Variable types: 0 continuous, 150 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [3e+04, 7e+05]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Found heuristic solution: objective 2.984847e+07
Presolve removed 22 rows and 48 columns
Presolve time: 0.00s
Presolved: 52 rows, 102 columns, 204 nonzeros
Variable types: 0 continuous, 102 integer (1 binary)
Found heuristic solution: objective 2.846780e+07

Root relaxation: objective 2.632722e+07, 53 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0   